# 05 — Analiz, Grafikler ve İstatistik (İP-7, İP-8)

Deney klasörlerindeki özet/soru-bazında CSV'lerden rapor tabloları ve grafikler üretir. Bu notebook GPU gerektirmez.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

REPO_URL = 'https://github.com/EmreK37/RAG-Tabanli-Turkce-Soru-Cevap-Sistemleri.git'
BRANCH = 'RAG-dev'
REPO_DIR = '/content/RAG-proje'
if os.path.exists('/content'):
    if not os.path.exists(REPO_DIR):
        !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}
    !git pull origin {BRANCH}

klasorler = sorted(Path('results').glob('ana-karsilastirma-*'))
son = klasorler[-1]
ozet = pd.read_csv(son / 'ozet.csv')
soru_yolu = son / 'soru-bazinda.csv'
soru = pd.read_csv(soru_yolu) if soru_yolu.exists() else None
print('Deney:', son)
ozet

In [ ]:
# Sistem karşılaştırma grafiği (rapor için PNG kaydeder)
fig, ax = plt.subplots(figsize=(8, 4))
ozet.plot.bar(x='sistem', y=['recall@10', 'ndcg@10', 'mrr@10'], ax=ax)
ax.set_ylabel('Skor')
ax.set_title('Retrieval karşılaştırması')
plt.tight_layout()
plt.savefig(son / 'grafik-retrieval.png', dpi=200)
plt.show()
print('Kaydedildi:', son / 'grafik-retrieval.png')

In [ ]:
# Soru tipine göre kırılım (RQ2): bicimbilimsel/paraphrase sorularda kayıp
if soru is not None:
    ndcg = soru[soru['metrik'] == 'ndcg@10']
    kirilim = ndcg.groupby(['sistem', 'tip'])['skor'].mean().unstack()
else:
    kirilim = pd.read_csv(son / 'rq2-kirilim.csv', index_col='sistem')

fig, ax = plt.subplots(figsize=(9, 4.5))
kirilim.plot.bar(ax=ax)
ax.set_ylabel('nDCG@10')
ax.set_title('Soru Tipine Göre Erişim Performansı (RQ2)')
ax.legend(title='Soru Tipi', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(son / 'grafik-rq2-kirilim.png', dpi=200)
plt.show()
kirilim

In [ ]:
# Maliyet özeti (RQ4) + RQ3 RAG Özeti + RQ5 Router Kalite-Maliyet Pareto Grafiği
print('=== RQ4: Maliyet ve Gecikme Özeti ===')
print(ozet[['sistem', 'indeks_suresi_sn', 'ort_gecikme_ms', 'ndcg@10', 'recall@5']].to_string(index=False))

if Path('results/rq3-rag-ozet.csv').exists():
    print('\n=== RQ3: RAG Yanıt Kalitesi ve Faithfulness Özeti ===')
    print(pd.read_csv('results/rq3-rag-ozet.csv').to_string(index=False))

if Path('results/rq5-router-ozet.csv').exists():
    rq5 = pd.read_csv('results/rq5-router-ozet.csv')
    print('\n=== RQ5: Sorguya Duyarlı Router ve Ablasyon Özeti ===')
    print(rq5.to_string(index=False))

    fig, ax = plt.subplots(figsize=(8, 4.5))
    for _, r in ozet.iterrows():
        ax.scatter(r['ort_gecikme_ms'], r['ndcg@10'], s=90, label=r['sistem'])
        ax.annotate(r['sistem'], (r['ort_gecikme_ms'] + 0.8, r['ndcg@10'] - 0.004))
    for _, r in rq5[rq5['sistem'].isin(['Router (RF - Tam)', 'Router (LogReg)'])].iterrows():
        ax.scatter(r['ort_gecikme_ms'], r['ndcg@10'], s=140, marker='*', label=r['sistem'])
        ax.annotate(r['sistem'], (r['ort_gecikme_ms'] + 0.8, r['ndcg@10'] + 0.004), fontweight='bold')
    ax.set_xlabel('Ortalama Sorgu Gecikmesi (ms)')
    ax.set_ylabel('nDCG@10')
    ax.set_title('Kalite - Gecikme Pareto Analizi (RQ4 & RQ5)')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(son / 'grafik-rq4-rq5-pareto.png', dpi=200)
    plt.show()

## Raporlama kontrol listesi (İP-8)

- [ ] Her tabloda kullanılan konfigürasyon ve tohumlar belirtilmiş
- [ ] Sistem çiftleri için eşleştirilmiş bootstrap güven aralıkları eklenmiş
- [ ] Soru tipine göre kırılım (RQ2) ve maliyet analizi (RQ4) dahil
- [ ] Hata analizi: en kötü 10 soru elle incelenip sınıflandırılmış
- [ ] Sonuç klasörleri Drive'a yedeklenmiş ve repoya özet CSV'ler işlenmiş